# One-signature signing-path PRF: chained CPA

This notebook matches the paper's **single-signature path** experiment:
3 fixed SK.seed values × 3 distinct signatures for each SHA2 parameter set,
with FORS and WOTS+ addresses analyzed separately. Each trial uses one
signature's PRF waveforms; no different signatures/keys are merged. Previous
*recovered* words (not true preceding words) are propagated. Set `RUN_CPA=True`
only after inspecting the local captures. Very large `s` sets need substantial
memory and disk space. This is **not** continuous full-signature acquisition.

The `PK_SEED_HEX` override is essential when local captures were obtained with
any non-demo PK.seed; do not commit the original experiment's key values.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory")
sys.path.insert(0, str(REPO_ROOT / "src"))

import signature_cpa_batch as runner


## Configuration

Check the dataset path and keep `RUN_CPA = False` until ready.


In [ ]:
BASE_FOLDER = Path(os.environ.get("SLH_CPA_SIGNATURE_BASE", "./data/signature_sequences"))
RESULTS_DIR = Path(os.environ.get("SLH_CPA_SIGNATURE_RESULTS", "./results/signature"))
PARAM_SETS = ("128f", "128s", "192f", "192s", "256f", "256s")
SEED_IDS = "0,1,2"
SIGNATURE_IDS = "0,1,2"
COMPILER_OPTS = ("Os",)
PK_SEED_HEX = os.environ.get("SLH_CPA_PK_SEED_HEX", "")
BACKEND = "auto"
RUN_CPA = False


## Data inventory and optional CPA execution

Dry runs inspect local measured datasets; they do not generate new measurements.


In [ ]:
def argv(output):
    args = ["--base", str(BASE_FOLDER), "--output", str(output),
            "--params", ",".join(PARAM_SETS), "--opts", ",".join(COMPILER_OPTS),
            "--source", "message", "--seed-ids", SEED_IDS,
            "--signature-ids", SIGNATURE_IDS, "--kinds", "fors,wots",
            "--attacks", "expansion,compression", "--protocol", "chained",
            "--per-signature", "auto", "--backend", BACKEND,
            "--beam", "4", "--joint-classes", "4", "--no-summary-plots"]
    if PK_SEED_HEX:
        args += ["--pk-seed-hex", PK_SEED_HEX]
    return args

if not BASE_FOLDER.is_dir():
    print("Missing local capture folder:", BASE_FOLDER)
else:
    print("Preflight:", runner.main(argv(RESULTS_DIR / "_preview") + ["--dry-run"]))
    if RUN_CPA:
        print("CPA:", runner.main(argv(RESULTS_DIR)))
    else:
        print("Preview only. Set RUN_CPA=True after checking the dataset and parameters.")
